# Clasificación binaria del abandono de clientes en telecomunicaciones

El objetivo de este trabajo es construir y comparar modelos de clasificación binaria para predecir la variable `Y`, que indica si un cliente abandona el servicio. El análisis se organiza en dos enfoques: RFE con regresión logística y red neuronal orientada a Accuracy, y árbol de decisión con red neuronal orientada a Recall.

## Plan de trabajo

1. Carga y revisión inicial de datos.
2. Análisis exploratorio y depuración.
3. Preparación metodológica: train/test y preprocesado.
4. Apartado 2: RFE con regresión logística + red neuronal orientada a Accuracy.
5. Apartado 3: árbol de decisión + red neuronal orientada a Recall.
6. Selección de thresholds y comparación global.
7. Conclusiones y material para memoria.

## 1. Carga de librerías y datos

Se cargan las librerías y la base original sin aplicar todavía transformaciones. Esta fase solo verifica que los datos se han leído correctamente.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix
)

In [ ]:
ruta_datos = "../data/BBDD_ML_TAREA.csv"
df = pd.read_csv(ruta_datos)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
df["Y"].value_counts()

### Observaciones iniciales

La base original contiene 9200 observaciones y 21 columnas: 20 variables explicativas (`V1` a `V20`) y la variable objetivo `Y`. En esta versión original, `Y` aparece balanceada con 4600 observaciones por clase. No se aplica todavía ninguna transformación, ya que antes se revisan nulos, duplicados, variables identificativas y redundancias.

## 2. Análisis exploratorio y depuración

Esta sección depura únicamente aquello que debe resolverse antes de la partición train/test: duplicados exactos, duplicados lógicos e identificación de variables que no deben entrar al modelado.

### 2.1 Valores perdidos

In [ ]:
nulos = pd.DataFrame({
    "n_nulos": df.isna().sum(),
    "porcentaje": df.isna().mean() * 100
}).sort_values("n_nulos", ascending=False)

nulos[nulos["n_nulos"] > 0]

Los valores perdidos afectan a pocas variables y siempre por debajo del 1%. No se imputan en esta fase: la imputación se integrará posteriormente en los pipelines y se ajustará solo con entrenamiento para evitar fuga de información.

### 2.2 Duplicados exactos

In [ ]:
resumen_duplicados = pd.Series({
    "filas_originales": len(df),
    "duplicados_exactos": df.duplicated().sum(),
    "filas_en_grupos_duplicados": df.duplicated(keep=False).sum(),
    "filas_unicas": len(df.drop_duplicates())
})

resumen_duplicados

In [ ]:
df_unico = df.drop_duplicates().reset_index(drop=True)

In [ ]:
pd.DataFrame({
    "original": df["Y"].value_counts().sort_index(),
    "sin_duplicados_exactos": df_unico["Y"].value_counts().sort_index()
})

Se eliminan duplicados exactos antes del split para evitar que copias idénticas puedan aparecer simultáneamente en entrenamiento y test. Tras esta depuración, la clase positiva deja de estar artificialmente balanceada, lo que confirma que las repeticiones afectaban de forma desigual a las clases.

### 2.3 Duplicados lógicos asociados a `V4`

In [ ]:
duplicados_v4 = df_unico[df_unico["V4"].duplicated(keep=False)].sort_values("V4")
duplicados_v4

In [ ]:
duplicados_v4.assign(n_nulos=duplicados_v4.isna().sum(axis=1))

In [ ]:
df_limpio = (
    df_unico
    .assign(n_nulos=df_unico.isna().sum(axis=1))
    .sort_values(["V4", "n_nulos"])
    .drop_duplicates(subset="V4", keep="first")
    .drop(columns="n_nulos")
    .reset_index(drop=True)
)

In [ ]:
pd.Series({
    "filas": df_limpio.shape[0],
    "columnas": df_limpio.shape[1],
    "duplicados_v4": df_limpio["V4"].duplicated().sum()
})

In [ ]:
df_limpio["Y"].value_counts()

In [ ]:
df_limpio["Y"].value_counts(normalize=True)

Los duplicados lógicos por `V4` corresponden a registros equivalentes en los que una versión contiene algún valor perdido adicional. Para cada `V4` repetido se conserva la fila más completa. Desde este punto, `df_limpio` es la base depurada de referencia.

### 2.4 Clasificación metodológica de variables

In [ ]:
variable_objetivo = "Y"

variables_identificativas = ["V4"]
variables_categoricas_codificadas = ["V1", "V3"]
variables_binarias = ["V5", "V6"]
variables_numericas = [
    "V2", "V7", "V8", "V9", "V10",
    "V11", "V12", "V13", "V14", "V15",
    "V16", "V17", "V18", "V19", "V20"
]

variables_clasificadas = (
    variables_identificativas
    + variables_categoricas_codificadas
    + variables_binarias
    + variables_numericas
    + [variable_objetivo]
)

set(df_limpio.columns) - set(variables_clasificadas), set(variables_clasificadas) - set(df_limpio.columns)

`V4` se considera identificativa y no se usará como predictor. `V1` y `V3` representan categorías codificadas mediante números, por lo que se tratarán mediante variables dummy en los pipelines. `V5` y `V6` son binarias; el resto son variables cuantitativas de antigüedad, consumo, llamadas, costes y atención al cliente.

### 2.5 Redundancia entre variables de minutos y coste

In [ ]:
pares_redundantes = [
    ("V8", "V10"),
    ("V11", "V13"),
    ("V14", "V16"),
    ("V17", "V19")
]

for var_minutos, var_coste in pares_redundantes:
    correlacion = df_limpio[[var_minutos, var_coste]].corr().iloc[0, 1]
    print(f"{var_minutos}-{var_coste}: {correlacion:.6f}")

In [ ]:
variables_minutos_costes = [
    "V8", "V10", "V11", "V13",
    "V14", "V16", "V17", "V19"
]

df_limpio[variables_minutos_costes].corr()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(
    df_limpio[variables_minutos_costes].corr(),
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)
plt.title("Correlación entre variables de minutos y costes")
plt.show()

Las variables de coste (`V10`, `V13`, `V16`, `V19`) están casi perfectamente correlacionadas con sus variables de minutos correspondientes. Para reducir redundancia y mejorar la estabilidad de la selección de variables, se conservan las variables de minutos y se excluyen las de coste.

### 2.6 Base preparada para modelado

In [ ]:
variables_coste_redundantes = ["V10", "V13", "V16", "V19"]
variables_excluidas = variables_identificativas + variables_coste_redundantes

df_modelado = df_limpio.drop(columns=variables_excluidas)

In [ ]:
df_modelado.shape

In [ ]:
df_modelado.columns.tolist()

In [ ]:
df_modelado.isna().sum()[df_modelado.isna().sum() > 0]

`df_modelado` excluye la variable identificativa y los costes redundantes. Los nulos restantes no se imputan todavía: se tratarán dentro de los pipelines tras la partición train/test.

## 3. Preparación metodológica antes del modelado

Se separa la variable objetivo, se realiza una partición estratificada y se definen transformaciones que se ajustarán exclusivamente sobre entrenamiento.

### 3.1 Separación `X` / `y`

In [ ]:
X = df_modelado.drop(columns="Y")
y = df_modelado["Y"]

In [ ]:
X.shape, y.shape

In [ ]:
y.value_counts(normalize=True)

Tras la depuración, la clase positiva queda como minoritaria. Por ello, la partición train/test se realiza de forma estratificada.

### 3.2 Partición train/test estratificada

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=21
)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
pd.DataFrame({
    "train": y_train.value_counts(normalize=True),
    "test": y_test.value_counts(normalize=True)
}).sort_index()

### 3.3 Preprocesado

In [ ]:
variables_categoricas = ["V1", "V3"]
variables_binarias = ["V5", "V6"]
variables_numericas_modelado = [
    "V2", "V7", "V8", "V9",
    "V11", "V12", "V14", "V15",
    "V17", "V18", "V20"
]

variables_preprocesado = variables_categoricas + variables_binarias + variables_numericas_modelado

set(X.columns) - set(variables_preprocesado), set(variables_preprocesado) - set(X.columns)

In [ ]:
transformador_categorico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

transformador_binario = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

transformador_numerico_escalado = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

transformador_numerico_sin_escalar = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

In [ ]:
preprocesador_lineal_nn = ColumnTransformer(
    transformers=[
        ("cat", transformador_categorico, variables_categoricas),
        ("bin", transformador_binario, variables_binarias),
        ("num", transformador_numerico_escalado, variables_numericas_modelado)
    ]
)

preprocesador_arbol = ColumnTransformer(
    transformers=[
        ("cat", transformador_categorico, variables_categoricas),
        ("bin", transformador_binario, variables_binarias),
        ("num", transformador_numerico_sin_escalar, variables_numericas_modelado)
    ]
)

La regresión logística y las redes neuronales usan imputación, dummies y escalado. El árbol usará imputación y dummies, pero no escalado, ya que no depende de distancias ni coeficientes lineales. En todos los casos, el preprocesado irá dentro de pipelines para evitar fuga de información.

## 4. Apartado 2: RFE con regresión logística y red neuronal orientada a Accuracy

### 4.1 RFE con regresión logística

RFE se aplica solo sobre entrenamiento. Como `V1` y `V3` generan dummies, primero se obtiene el ranking sobre columnas transformadas y después se interpreta a nivel de variable original.

In [ ]:
X_train_lineal_nn = preprocesador_lineal_nn.fit_transform(X_train)
X_test_lineal_nn = preprocesador_lineal_nn.transform(X_test)

nombres_features_lineal_nn = preprocesador_lineal_nn.get_feature_names_out()
X_train_lineal_nn.shape, X_test_lineal_nn.shape, len(nombres_features_lineal_nn)

In [ ]:
log_reg_base = LogisticRegression(
    max_iter=5000,
    solver="liblinear",
    class_weight="balanced",
    random_state=21
)

rfe_logistica = RFE(
    estimator=log_reg_base,
    n_features_to_select=5
)

rfe_logistica.fit(X_train_lineal_nn, y_train)

In [ ]:
features_rfe = pd.DataFrame({
    "feature": nombres_features_lineal_nn,
    "seleccionada": rfe_logistica.support_,
    "ranking": rfe_logistica.ranking_
}).sort_values(["ranking", "feature"])

features_rfe.head(15)

In [ ]:
features_seleccionadas_rfe = features_rfe[features_rfe["seleccionada"]]["feature"].tolist()
features_seleccionadas_rfe

El RFE inicial selecciona columnas transformadas. Varias pueden corresponder a distintas dummies de una misma variable original; por ello se agrega el ranking a nivel de variable de partida para obtener cinco variables originales interpretables.

In [ ]:
def obtener_variable_original(nombre_feature):
    nombre_sin_prefijo = nombre_feature.split("__", 1)[1]
    return nombre_sin_prefijo.split("_")[0]

features_rfe["variable_original"] = features_rfe["feature"].apply(obtener_variable_original)

ranking_variables_originales = (
    features_rfe
    .groupby("variable_original")
    .agg(
        mejor_ranking=("ranking", "min"),
        n_columnas_transformadas=("feature", "count"),
        n_columnas_seleccionadas=("seleccionada", "sum")
    )
    .sort_values(["mejor_ranking", "variable_original"])
)

ranking_variables_originales.head(10)

In [ ]:
variables_originales_seleccionadas_rfe = (
    ranking_variables_originales
    .head(5)
    .index
    .tolist()
)

variables_originales_seleccionadas_rfe

La selección final del apartado 2 se interpreta a nivel de variable original: `V1`, `V5`, `V6`, `V20` y `V8`. Si `V1` aparece por alguna de sus dummies, se conserva la variable original completa y se codifica de nuevo en el pipeline de la red neuronal.

### 4.2 Red neuronal con variables RFE

In [ ]:
X_train_rfe = X_train[variables_originales_seleccionadas_rfe]
X_test_rfe = X_test[variables_originales_seleccionadas_rfe]

X_train_rfe.shape, X_test_rfe.shape

In [ ]:
variables_categoricas_rfe = ["V1"]
variables_binarias_rfe = ["V5", "V6"]
variables_numericas_rfe = ["V20", "V8"]

preprocesador_nn_rfe = ColumnTransformer(
    transformers=[
        ("cat", transformador_categorico, variables_categoricas_rfe),
        ("bin", transformador_binario, variables_binarias_rfe),
        ("num", transformador_numerico_escalado, variables_numericas_rfe)
    ]
)

La red neuronal usa solo las cinco variables originales seleccionadas. `V1` se transforma mediante dummies, `V5` y `V6` entran como binarias, y `V20` y `V8` se imputan y escalan.

### 4.3 Búsqueda paramétrica orientada a Accuracy

In [ ]:
pipeline_nn_rfe = Pipeline(steps=[
    ("preprocesado", preprocesador_nn_rfe),
    ("modelo", MLPClassifier(
        max_iter=1000,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20,
        random_state=21
    ))
])

param_grid_nn_accuracy = {
    "modelo__hidden_layer_sizes": [(3,), (5,), (10,), (20,), (10, 5), (20, 10)],
    "modelo__activation": ["relu", "tanh"],
    "modelo__alpha": [0.0001, 0.001, 0.01],
    "modelo__learning_rate_init": [0.001, 0.01]
}

cv_accuracy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=21
)

grid_nn_accuracy = GridSearchCV(
    estimator=pipeline_nn_rfe,
    param_grid=param_grid_nn_accuracy,
    scoring="accuracy",
    cv=cv_accuracy,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

In [ ]:
grid_nn_accuracy.fit(X_train_rfe, y_train)

In [ ]:
grid_nn_accuracy.best_params_

In [ ]:
grid_nn_accuracy.best_score_

In [ ]:
resultados_grid_accuracy = pd.DataFrame(grid_nn_accuracy.cv_results_)

columnas_resultados = [
    "mean_train_score",
    "mean_test_score",
    "std_test_score",
    "param_modelo__hidden_layer_sizes",
    "param_modelo__activation",
    "param_modelo__alpha",
    "param_modelo__learning_rate_init"
]

resultados_grid_accuracy[columnas_resultados].sort_values(
    "mean_test_score",
    ascending=False
).head(10)

La búsqueda prueba 72 combinaciones: seis arquitecturas, dos funciones de activación, tres valores de regularización y dos tasas de aprendizaje. La mejor configuración obtiene una Accuracy media de validación cruzada aproximada de 0.859.

### 4.4 Selección del threshold

El threshold se selecciona usando probabilidades out-of-fold sobre entrenamiento. Así no se usa el conjunto de test para elegir el umbral.

In [ ]:
mejor_pipeline_nn_accuracy = grid_nn_accuracy.best_estimator_

proba_train_oof_accuracy = cross_val_predict(
    mejor_pipeline_nn_accuracy,
    X_train_rfe,
    y_train,
    cv=cv_accuracy,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

In [ ]:
thresholds = np.arange(0.1, 1.0, 0.05)

metricas_threshold_accuracy = []

for threshold in thresholds:
    y_pred_threshold = (proba_train_oof_accuracy >= threshold).astype(int)
    metricas_threshold_accuracy.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_train, y_pred_threshold),
        "recall": recall_score(y_train, y_pred_threshold),
        "precision": precision_score(y_train, y_pred_threshold, zero_division=0),
        "f1": f1_score(y_train, y_pred_threshold)
    })

metricas_threshold_accuracy = pd.DataFrame(metricas_threshold_accuracy)
metricas_threshold_accuracy

In [ ]:
metricas_threshold_accuracy.sort_values("accuracy", ascending=False).head(10)

In [ ]:
plt.figure(figsize=(9, 6))

plt.plot(metricas_threshold_accuracy["threshold"], metricas_threshold_accuracy["accuracy"], marker="o", label="Accuracy")
plt.plot(metricas_threshold_accuracy["threshold"], metricas_threshold_accuracy["recall"], marker="o", label="Recall")
plt.plot(metricas_threshold_accuracy["threshold"], metricas_threshold_accuracy["precision"], marker="o", label="Precision")
plt.plot(metricas_threshold_accuracy["threshold"], metricas_threshold_accuracy["f1"], marker="o", label="F1-score")

plt.xlabel("Threshold")
plt.ylabel("Métrica")
plt.title("Evolución de métricas según threshold - Red neuronal Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
threshold_accuracy = metricas_threshold_accuracy.loc[
    metricas_threshold_accuracy["accuracy"].idxmax(),
    "threshold"
]

threshold_accuracy

El threshold que maximiza Accuracy en validación out-of-fold es 0.50. Umbrales menores aumentan el Recall a costa de Accuracy y Precision; umbrales mayores incrementan Precision pero reducen la detección de la clase positiva.

### 4.5 Evaluación final en test

In [ ]:
modelo_final_nn_accuracy = grid_nn_accuracy.best_estimator_
proba_test_accuracy = modelo_final_nn_accuracy.predict_proba(X_test_rfe)[:, 1]
y_pred_test_accuracy = (proba_test_accuracy >= threshold_accuracy).astype(int)

In [ ]:
metricas_test_nn_accuracy = pd.DataFrame([{
    "modelo": "NN_RFE_Accuracy",
    "threshold": threshold_accuracy,
    "accuracy": accuracy_score(y_test, y_pred_test_accuracy),
    "recall": recall_score(y_test, y_pred_test_accuracy),
    "precision": precision_score(y_test, y_pred_test_accuracy, zero_division=0),
    "f1": f1_score(y_test, y_pred_test_accuracy)
}])

metricas_test_nn_accuracy

In [ ]:
matriz_confusion_nn_accuracy = confusion_matrix(y_test, y_pred_test_accuracy)
matriz_confusion_nn_accuracy

In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(
    matriz_confusion_nn_accuracy,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Pred 0", "Pred 1"],
    yticklabels=["Real 0", "Real 1"]
)
plt.title("Matriz de confusión - NN RFE Accuracy")
plt.xlabel("Predicción")
plt.ylabel("Valor real")
plt.show()

El modelo final del apartado 2 se evalúa una única vez en test con el threshold seleccionado en entrenamiento. El resultado final es Accuracy 0.839, Recall 0.433, Precision 0.642 y F1-score 0.517. La matriz de confusión muestra 533 verdaderos negativos, 34 falsos positivos, 80 falsos negativos y 61 verdaderos positivos.